Download packages

In [ ]:
#Need to install
#!pip install pandas numpy sklearn.metrics sklearn.model_selection emoji re datasets transformers warnings


In [1]:
# For data manipulation and analysis
import pandas as pd
import numpy as np

# For machine learning tools and evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, balanced_accuracy_score


from sklearn.model_selection import train_test_split

import emoji
import re
import datasets
from transformers import TrainingArguments, Trainer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import warnings
from scipy.stats import chisquare

c:\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Get tokenizer

In [2]:
tokenizer = AutoTokenizer.from_pretrained("DTAI-KULeuven/robbert-2022-dutch-base")

Get BERT model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("DTAI-KULeuven/robbert-2022-dutch-base")

Prepare data

In [ ]:
# Read data 
data =  pd.read_csv("Data.csv")
# Classification only works when outcome variable is called labels
data[['labels']] = data[['label_mis']]
# Convert emoji to descriptions (in English) using emoji package
def no_emoji(text):
    text = emoji.demojize(text) 
    return text 
data['text'] = data['text'].apply(no_emoji)
#Remove urls, &gt, &lt and &amp, and [numbers] from text
#Removed [numbers] because there were meaningless numbers between brackets in the Tweets
def clean_text(text):
    text = re.sub(r'https?://\S+|www\.\S+|\r|\n|&gt.?| &lt.?|&amp.?|\[\d*\]', '', text)
    return text 
data['text'] = data['text'].apply(clean_text) 

#Display cleaned text
pd.set_option('display.max_rows', None)
pd.set_option('max_colwidth', None)

#One dataset with only text and labels to train BERT model
#One dataset with also year so I can later split dataset and evaluate metrics per year
data_inf = data[['text', 'labels', 'id', 'year']]
data = data[['text', 'labels']]



Set values of variables important for training

In [ ]:
max_length = 512
# set random seed for reproducibility
SEED_GLOBAL = 42
np.random.seed(SEED_GLOBAL)
#Set training directory 
training_directory = "Roberta"

Split data into training and test set

In [4]:
# Train and test set for training model
df_train, df_test = train_test_split(data, random_state=42, test_size=0.25)

#Create identical test with text and labels plus year so performance metrics can be split by year
df_train_inf, df_test_inf = train_test_split(data_inf, random_state=42, test_size=0.25)

Tokenize data (from https://github.com/MoritzLaurer/summer-school-transformers-2023/blob/main/3_tune_bert.ipynb)

In [5]:
# convert pandas dataframes to Hugging Face dataset object to facilitate pre-processing
dataset = datasets.DatasetDict({
    "train": datasets.Dataset.from_pandas(df_train),
    "test": datasets.Dataset.from_pandas(df_test)
})

# tokenize
def tokenize(examples):
  return tokenizer(examples["text"], truncation=True, padding = 'max_length', max_length=512)  # max_length can be reduced to e.g. 256 to increase speed, but long texts will be cut off

dataset = dataset.map(tokenize, batched=True)

Map: 100%|██████████| 849/849 [00:00<00:00, 4081.92 examples/s]


Set training arguments (from https://github.com/MoritzLaurer/summer-school-transformers-2023/blob/main/3_tune_bert.ipynb)

In [35]:
train_args = TrainingArguments(
    num_train_epochs=3,  
    learning_rate=2e-5,
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=64,   
    warmup_ratio=0.06, 
    weight_decay=0.1,
    seed=SEED_GLOBAL,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    evaluation_strategy="epoch", 
    save_strategy = "epoch",
    report_to="all",
    output_dir=f'{training_directory}',
    logging_dir=f'{training_directory}',
)


Set evaluation metrics (from https://github.com/MoritzLaurer/summer-school-transformers-2023/blob/main/3_tune_bert.ipynb)

In [36]:
# Function to calculate metrics
# documentation on all metrics: https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics

def compute_metrics_standard(eval_pred):
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")

        labels = eval_pred.label_ids
        pred_logits = eval_pred.predictions
        preds_max = np.argmax(pred_logits, axis=1) 

        # metrics
        precision_mis, recall_mis, f1_mis, _ = precision_recall_fscore_support(labels, preds_max, average= 'binary') 
        precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(labels, preds_max, average='macro')  
        precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(labels, preds_max, average='micro')  
        acc_balanced = balanced_accuracy_score(labels, preds_max)
        acc_not_balanced = accuracy_score(labels, preds_max)

        metrics = {
            'accuracy': acc_not_balanced,
            'f1_macro': f1_macro,
            'accuracy_balanced': acc_balanced,
            'f1_micro': f1_micro,
            'precision_macro': precision_macro,
            'recall_macro': recall_macro,
            'precision_micro': precision_micro,
            'recall_micro': recall_micro,
            'precision_misinformation': precision_mis,
            'recall_misinformation': recall_mis,
            'f1_misinformation': f1_mis,
        }

        return metrics

Set BERT model

In [37]:
trainer = Trainer(
    model=model,                         
    args=train_args,             
    train_dataset=dataset["train"],        
    eval_dataset=dataset["test"],           
    compute_metrics=compute_metrics_standard     
)

  4%|▍         | 18/480 [47:10<20:10:44, 157.24s/it]


Train BERT model 

In [ ]:
trainer.train()

Save model

In [ ]:
trainer.save_model(output_dir = "\\Robbert_2022\\Safe")

Load model

In [ ]:
model_path = "Robbert_2022\\Safe"
model = AutoModelForSequenceClassification.from_pretrained(model_path)

Seeing what model predicts as misinformation (from https://github.com/MoritzLaurer/summer-school-transformers-2023/blob/main/3_tune_bert.ipynb)

In [7]:
from transformers import pipeline

# documentation: https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.ZeroShotClassificationPipeline
pipe_classifier = pipeline(
    "text-classification",
    model=model,  
    tokenizer=tokenizer,
    framework="pt"
)

Device set to use cpu


Inference: inspect what BERT model classifies as misinformation

In [ ]:
#Use df_test with date so data can be splitted on year
df_inference = df_test_inf[["text", "labels", "id", "year"]].copy(deep=True)
text_lst = df_inference["text"].tolist()

#inference
pipe_output = pipe_classifier(
    text_lst, 
    batch_size=64 
)
print(pipe_output)

df_output = pd.DataFrame(pipe_output)

# add inference data to original dataframe
df_inference["label_text_pred"] = df_output["label"].tolist()
df_inference["label_text_pred_probability"] = df_output["score"].round(2).tolist()
#Print df_inference
df_inference 

Calculate metrics per year

Split dataset on date

In [9]:
# Convert 'label_text_pred' to binary values
df_inference['label_pred'] = df_inference['label_text_pred'].apply(lambda x: 1 if x == 'LABEL_1' else 0)

#Split data
joint_data_2020 = df_inference[df_inference['year'] == 2020]
joint_data_2021 = df_inference[df_inference['year'] != 2020]

Calculate recall and precision for overall dataset and datasets split on year

In [13]:
# Calculate precision and recall overall (to test if this gets me the same results as above)
precision = precision_recall_fscore_support(y_true=df_inference['labels'], y_pred=df_inference['label_pred'], average='macro')
#recall = recall_score(df_inference['labels'], df_inference['label_pred'], average= 'binary')
print(f'Precision: {precision}')
#print(f'Recall: {recall}')

Precision: (0.7850869280898369, 0.7125832324455206, 0.7381765761975227, None)


In [14]:
# Calculate precision and recall (2020)
precision_recall_f = precision_recall_fscore_support(y_true=joint_data_2020['labels'], y_pred=joint_data_2020['label_pred'], average='binary')
precision_recall_f_macro = precision_recall_fscore_support(y_true=joint_data_2020['labels'], y_pred=joint_data_2020['label_pred'], average='macro')
accuracy = accuracy_score(y_true=joint_data_2020['labels'], y_pred=joint_data_2020['label_pred'])
print(f'Precision_recall_fscore: {precision_recall_f}')
print(f'Precision_recall_fscore_macro: {precision_recall_f_macro}')
print(f'Accuracy: {accuracy}')

Precision_recall_fscore: (0.6382978723404256, 0.42857142857142855, 0.5128205128205128, None)
Precision_recall_fscore_macro: (0.7707227860491475, 0.6924908424908425, 0.7209183510553373, None)
Accuracy: 0.8760869565217392


In [15]:
# Calculate precision and recall (after 2020)
precision_recall_f = precision_recall_fscore_support(y_true=joint_data_2021['labels'], y_pred=joint_data_2021['label_pred'], average='binary')
precision_recall_f_macro = precision_recall_fscore_support(y_true=joint_data_2021['labels'], y_pred=joint_data_2021['label_pred'], average='macro')
accuracy = accuracy_score(y_true=joint_data_2021['labels'], y_pred=joint_data_2021['label_pred'])
print(f'Precision_recall_fscore: {precision_recall_f}')
print(f'Precision_recall_fscore_macro: {precision_recall_f_macro}')
print(f'Accuracy: {accuracy}')

Precision_recall_fscore: (0.7333333333333333, 0.514018691588785, 0.6043956043956042, None)
Precision_recall_fscore_macro: (0.7838641188959661, 0.7215483528865911, 0.7417951176340437, None)
Accuracy: 0.8149100257069408


Calculate metrics overall and per year for accurate information Tweets

In [16]:
precision = precision_recall_fscore_support(y_true=df_inference['labels'], y_pred=df_inference['label_pred'], average='binary', pos_label =0)
print(f'Precision, recall, fscore: {precision}')

Precision, recall, fscore: (0.8734525447042641, 0.9449404761904762, 0.9077912794853467, None)


In [17]:
# Calculate precision and recall (2020)
precision_recall_f = precision_recall_fscore_support(y_true=joint_data_2020['labels'], y_pred=joint_data_2020['label_pred'], average='binary', pos_label =0)
print(f'Precision_recall_fscore: {precision_recall_f}')

Precision_recall_fscore: (0.9031476997578692, 0.9564102564102565, 0.9290161892901618, None)


In [18]:
# Calculate precision and recall (after 2020)
precision_recall_f = precision_recall_fscore_support(y_true=joint_data_2021['labels'], y_pred=joint_data_2021['label_pred'], average='binary', pos_label =0)
print(f'Precision_recall_fscore: {precision_recall_f}')

Precision_recall_fscore: (0.8343949044585988, 0.9290780141843972, 0.8791946308724832, None)


Chi square tests

In [10]:
#merge 2021 and 2022 in data inference dataset
df_inference['year'] = df_inference['year'].apply(lambda x: 2020 if x == 2020 else 2021)

In [11]:
#calculate function for whether labels were correctly predicted
#Make recall variable
#1 means model correctly predicted label
# 0 means did not correctly predict label
def num(row):
    if row['labels'] == row['label_pred']:
        return 1
    else : 
        return 0

df_inference['correct'] = df_inference.apply(num, axis =1)

Recall misinformation

In [12]:

#Select only misinformation variable (to calculate significance for recall misinformation variable, not accuracy)
df_inf_mis = df_inference[df_inference["labels"]==1]


#make table of number of tweets in each group
df_inf_mis.groupby(['correct', 'year']).size()

correct  year
0        2020    40
         2021    52
1        2020    30
         2021    55
dtype: int64

In [ ]:
#Test difference between 2020 and 2021 
chisquare([40, 52, 30, 55])

Recall accurate information

In [60]:
#Select only accurate information variable (to calculate significance for recall accurate information)
df_inf_mis = df_inference[df_inference["labels"]==0]


#make table of number of tweets in each group
df_inf_mis.groupby(['correct', 'year']).size()

correct  year
0        2020     17
         2021     20
1        2020    373
         2021    262
dtype: int64

In [61]:
#Test difference between 2020 and 2021 
chisquare([17, 20, 373, 262])

Power_divergenceResult(statistic=568.8452380952381, pvalue=5.715110185418486e-123)

Accuracy

In [52]:
df_inference.groupby(['correct','year']).size()


correct  year
0        2020     57
         2021     72
1        2020    403
         2021    317
dtype: int64

In [56]:
chisquare([57, 72, 403, 317])

Power_divergenceResult(statistic=429.3557126030624, pvalue=9.681625917453196e-93)

Precision misinformation

In [ ]:
#Select only predicted misinformation (to calculate significance for precision misinformation label)
df_inf_mis = df_inference[df_inference["label_pred"]==1]
#make table of number of tweets in each group
df_inf_mis.groupby(['correct', 'year']).size()

correct  year
0        2020    17
         2021    20
1        2020    30
         2021    55
dtype: int64

In [59]:
chisquare([17,20,30,55])

Power_divergenceResult(statistic=29.278688524590166, pvalue=1.9568970247577142e-06)

precision accurate information

In [62]:
#Select only predicted accurate information (to calculate significance for precision accurate information)
df_inf_mis = df_inference[df_inference["label_pred"]==0]
#make table of number of tweets in each group
df_inf_mis.groupby(['correct', 'year']).size()

correct  year
0        2020     40
         2021     52
1        2020    373
         2021    262
dtype: int64

In [63]:
chisquare([40,52,373,262])

Power_divergenceResult(statistic=439.8610729023384, pvalue=5.1281821523344916e-95)